# 1. Data Cleaning



In [132]:
import pandas as pd
import numpy as np
import re

**HELPER FUNCTIONS**

In [133]:
def parse_int(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace("\xa0", " ").replace(" ", "")
    nums = re.findall(r"\d+", s)
    if not nums:
        return np.nan
    return int("".join(nums))

def parse_float(x):
    if pd.isna(x):
          return np.nan
    try:
        first = str(x).split()[0].replace(",", ".")
        return float(first)
    except:
        return np.nan


def parse_engine_liters(x):
    if pd.isna(x):
        return np.nan

    try:
        val, unit = str(x).split()
        val = float(val.replace(",", "."))
    except:
        return np.nan

    unit = unit.lower()
    if unit.startswith("l"):
        return val
    if unit.startswith("cm"):
        return round(val / 1000, 1)

    return np.nan

def parse_power(x):
    if pd.isna(x):
        return np.nan

    try:
        value, unit = str(x).split()
        value = float(value.replace(",", "."))
    except:
        return np.nan

    unit = unit.lower()

    if unit == "kw":
        return int(round(value))

    if unit == "hj":
        return int(round(value * 0.7355))

    return np.nan

**READING IN DATA**

In [134]:
csv_files = [
    "auto24_data(1st batch).csv",
    "auto24_data(2nd batch).csv",
    "auto24_data(3rd batch).csv",
    "auto24_data(last batch).csv",
]

dfs = []
for f in csv_files:
    print("Reading:", f)
    if f.endswith("(2nd batch).csv"):
      df_part = pd.read_csv(
          f,
          usecols=range(276),
          on_bad_lines="skip"
      )

    else:
      df_part = pd.read_csv(
          f,
          engine="python",       # more tolerant parser
          on_bad_lines="skip"    # skip rows with wrong number of columns
    )
    print(f, "shape:", df_part.shape)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

print("Rows after merge:", df.shape)

Reading: auto24_data(1st batch).csv
auto24_data(1st batch).csv shape: (4000, 272)
Reading: auto24_data(2nd batch).csv
auto24_data(2nd batch).csv shape: (4000, 276)
Reading: auto24_data(3rd batch).csv


/tmp/ipython-input-3787720365.py:12: DtypeWarning: Columns (257,262,263) have mixed types. Specify dtype option on import or set low_memory=False.
  df_part = pd.read_csv(


auto24_data(3rd batch).csv shape: (3000, 275)
Reading: auto24_data(last batch).csv
auto24_data(last batch).csv shape: (309, 244)
Rows after merge: (11309, 521)


**REMOVE DUPLICANTS**

In [135]:
if "Link" in df.columns:
    df = df.drop_duplicates(subset="Link")

print("Rows after removing duplicates:", df.shape)

Rows after removing duplicates: (10964, 521)


**COMBINING COLUMNS**

In [136]:
df_copy = df.copy()
# mitu veergu enne merge'i

cols = [
    ["kesklukustus","kesklukustus (puldiga)"],
    ["isofix lasteistme kinnitus", "isofix lasteistme kinnitus (ees)", "isofix lasteistme kinnitus (taga)", "isofix lasteistme kinnitus (ees, taga)"],
    ["xenon", "xenon (lähituled)", "xenon (kaugtuled)", "xenon (lähituled, kaugtuled)"],
    ["led", 'led (päevatuled)', 'led (päevatuled, tagatuled)', 'led (päevatuled, tagatuled, lähituled)',
     'led (päevatuled, tagatuled, lähituled, kaugtuled)', 'led (tagatuled)', 'led (tagatuled, lähituled)',
    'led (tagatuled, lähituled, kaugtuled)', 'led (lähituled)', 'led (lähituled, kaugtuled)', 'led (kaugtuled)'],
    ["udutuled", 'udutuled (eesmised)', 'udutuled (eesmised, tagumine)', 'udutuled (tagumine)'],
    ['udutuled (kurvitule funktsiooniga)', 'udutuled (tagumine, kurvitule funktsiooniga)', 'udutuled (eesmised, tagumine, kurvitule funktsiooniga)'],
    ["reguleeritav roolisammas", 'reguleeritav roolisammas (kõrgus ja sügavus)'],
    ['reguleeritav roolisammas (elektriliselt)', 'reguleeritav roolisammas (kõrgus ja sügavus, elektriliselt)','reguleeritav roolisammas (kõrgus ja sügavus, elektriliselt, mäluga)',
    'reguleeritav roolisammas (elektriliselt, mäluga)', 'reguleeritav roolisammas (mäluga)'],
    ["navigatsiooniseade", 'navigatsiooniseade (kaardiga)', 'navigatsiooniseade (kaardiga, hääljuhtimisega)', 'navigatsiooniseade (hääljuhtimisega)'],
    ["jalamatid", 'jalamatid (tekstiilist)', 'jalamatid (tekstiilist, kummist)', 'jalamatid (tekstiilist, kummist, veluurist)',
    'jalamatid (kummist)', 'jalamatid (kummist, veluurist)', 'jalamatid (veluurist)'],
    ["topsihoidjad", "topsihoidjad (ees, taga)", "topsihoidjad (taga)", "topsihoidjad (ees)"],
    ["istmed reguleeritava kõrgusega", "istmed reguleeritava kõrgusega (juhiiste, kõrvalistuja iste)", "istmed reguleeritava kõrgusega (juhiiste)", "istmed reguleeritava kõrgusega (kõrvalistuja iste)"],
    ["käetugi ees", "käetugi ees (laekaga)"],
    ["käetugi taga", "käetugi taga (laekaga)"],
    ["elektrilised välispeeglid", 'elektrilised välispeeglid (soojendusega)','elektrilised välispeeglid (soojendusega, kokkuklapitavad)','elektrilised välispeeglid (kokkuklapitavad)'],
    ['elektrilised välispeeglid (mäluga)', 'elektrilised välispeeglid (soojendusega, kokkuklapitavad, mäluga)','elektrilised välispeeglid (kokkuklapitavad, mäluga)'],
    ["pakiruumi avamine elektriliselt", "pakiruumi avamine elektriliselt (puldist)"],
    ["pakiruumi avamine elektriliselt (jalaviipega)", "pakiruumi avamine elektriliselt (puldist, jalaviipega)"],
    ["rulookardin tagaaknal", "rulookardin tagaaknal (elektriline)"],
    ["automaatselt tumenevad peeglid", "automaatselt tumenevad peeglid (sees, väljas)", "automaatselt tumenevad peeglid (sees)", "automaatselt tumenevad peeglid (väljas)", "parkimisandurid", "parkimisandurid (ees, taga)"],
    ["reguleeritav vedrustus",'reguleeritav vedrustus (elektriliselt)', 'reguleeritav vedrustus (elektriliselt, jäikus)','reguleeritav vedrustus (elektriliselt, jäikus, kõrgus)', 'reguleeritav vedrustus (jäikus)',
    'reguleeritav vedrustus (jäikus, kõrgus)', 'reguleeritav vedrustus (kõrgus)'],
    ["pagasikate", "pagasikate (automaatne)"],
    ["veokonks", "veokonks (teisaldatav)"],
    ["veokonks (elektriline)", "veokonks (elektriline, teisaldatav)"]
    ]
for c in cols:
  main_column = c[0]
  df_copy[main_column] = df_copy[c].fillna(0).astype(int).any(axis=1).astype(int)
  df_copy = df_copy.drop(columns=c[1:])



print("Size before: ", df.shape, "\nSize after: ", df_copy.shape)
df = df_copy



Size before:  (10964, 521) 
Size after:  (10964, 455)


**ADDON PROCESSING**

In [137]:
#Identify addon columns
col_list = list(df.columns)
start_idx = col_list.index("abs pidurid")
end_idx = col_list.index("Liik")
addon_cols = col_list[start_idx:end_idx]

for c in addon_cols:
    df[c] = df[c].notna().astype(int)

**DROPING UNNECESSARY COLUMNS**

In [138]:
before = df.shape[1]
last_col = col_list.index("sildade arv")
df = df.iloc[:, :last_col + 1]
after = df.shape[1]
print("Columns before:", before, "\tColumns after:", after)
print("Removed ", before - after, " columns.")

Columns before: 455 	Columns after: 189
Removed  266  columns.


In [139]:
df.columns = df.columns.str.strip().str.replace("\xa0", "", regex=False)

**CLEAN KEY COLUMNS**

In [140]:
# Numeric columns
shape_before = df.shape
df["price"] = df["Hind"].apply(parse_int)
df["mileage_km"] = df["Läbisõidumõõdiku näit"].apply(parse_int)
df["reg_year"] = df["Esmane reg"].astype(str).str.extract(r"(\d{4})").astype(float)
df["engine_l"] = df["mootori maht"].apply(parse_engine_liters)
df["power_kw"] = df["võimsus"].apply(parse_power)
df["top_speed"] = df["tippkiirus"].apply(parse_int)
df["fuel_cons_city"] = df["- linnas"].apply(parse_float)
df["fuel_cons_highway"] = df["- maanteel"].apply(parse_float)
df["fuel_cons_avg"] = df["- keskmine"].apply(parse_float)
if "istekohti" in df.columns:
    df["seats"] = df["istekohti"].astype(float)
if "uste arv" in df.columns:
    df["doors"] = df["uste arv"].astype(float)

# Text columns to one-hot vectors
df["Mark"] = df["Mark"].str.strip()
df = pd.get_dummies(df, columns=["Mark"], prefix="make")
df = pd.get_dummies(df, columns=["Liik"], prefix="type")
df["Keretüüp"] = df["Keretüüp"].str.strip().str.split().str[0]
df = pd.get_dummies(df, columns=["Keretüüp"], prefix="bodytype")
df = pd.get_dummies(df, columns=["Kütus"], prefix="fuel")
df = pd.get_dummies(df, columns=["Vedav sild"], prefix="drivetrain")
df["Käigukast"] = df["Käigukast"].str.strip().str.split().str[0]
df = pd.get_dummies(df, columns=["Käigukast"], prefix="gearbox")

start = df.columns.get_loc("Esmane reg")
end = df.columns.get_loc("sildade arv")

df = df.drop(columns=df.columns[start : end + 1])
df = df.drop(columns=["Täisnimi"])
shape_after = df.shape
print("Shape before:", shape_before, "\tShape after:", shape_after)

/tmp/ipython-input-4128353649.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["price"] = df["Hind"].apply(parse_int)
/tmp/ipython-input-4128353649.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["mileage_km"] = df["Läbisõidumõõdiku näit"].apply(parse_int)
/tmp/ipython-input-4128353649.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de

Shape before: (10964, 189) 	Shape after: (10964, 283)


In [141]:
rare_cols = [c for c in df.columns if df[c].dtype != "object" and df[c].sum() < 10]
df.drop(columns=rare_cols, inplace=True)

**FILTER BROKEN ROWS**

In [142]:
print("=== DEBUG: BEFORE FILTERING ===")
print("Total rows before filtering:", df.shape[0])

print("Missing price:", df["price"].isna().sum())
print("Missing mileage_km:", df["mileage_km"].isna().sum())
print("Missing reg_year:", df["reg_year"].isna().sum())
print("Missing power_kw:", df["power_kw"].isna().sum())

print("Price min/max:", df["price"].min(), df["price"].max())
print("Mileage min/max:", df["mileage_km"].min(), df["mileage_km"].max())
print("Year min/max:", df["reg_year"].min(), df["reg_year"].max())
print("Power min/max", df["power_kw"].min(), df["power_kw"].max())

=== DEBUG: BEFORE FILTERING ===
Total rows before filtering: 10964
Missing price: 4
Missing mileage_km: 325
Missing reg_year: 28
Missing power_kw: 171
Price min/max: 340.0 579000.0
Mileage min/max: 1.0 3200000.0
Year min/max: 1949.0 2025.0
Power min/max 6.0 750.0


In [143]:
before = df.shape[0]

df = df[
    df["price"].notna() &
    df["mileage_km"].notna() &
    df["reg_year"].notna() &
    df["power_kw"].notna()

]

df = df[(df["reg_year"] >= 1990) & (df["reg_year"] <= 2025)]
df = df[(df["mileage_km"] >= 0) & (df["mileage_km"] <= 800000)]
df = df[(df["price"] >= 300) & (df["price"] <= 300000)]

after = df.shape[0]

print(f"Removed {before - after} invalid rows.")
print("Remaining rows:", after)

Removed 605 invalid rows.
Remaining rows: 10359


**FEATURE ENGINEERING**

In [144]:


CURRENT_YEAR = 2025
df["car_age"] = CURRENT_YEAR - df["reg_year"]
df.loc[df["car_age"] < 0, "car_age"] = np.nan

df["price_per_km"] = df["price"] / df["mileage_km"].replace({0: np.nan})
df["price_per_kw"] = df["price"] / df["power_kw"].replace({0: np.nan})
df["log_price"] = np.log1p(df["price"])

# SAVE

df.to_csv("auto24_cleaned.csv", index=False)
print("Saved cleaned dataset as auto24_cleaned.csv")

Saved cleaned dataset as auto24_cleaned.csv
